In [1]:
!pip install sentence-transformers


In [2]:
!pip install git+https://github.com/daviddavo/lightfm

  Cloning https://github.com/daviddavo/lightfm to /tmp/pip-req-build-7sxn4vik
  Running command git clone --filter=blob:none --quiet https://github.com/daviddavo/lightfm /tmp/pip-req-build-7sxn4vik
  Resolved https://github.com/daviddavo/lightfm to commit f0eb500ead54ab65eb8e1b3890337a7223a35114
  Preparing metadata (setup.py) ... done
  Created wheel for lightfm: filename=lightfm-1.17-cp312-cp312-linux_x86_64.whl size=1100854 sha256=047e8c8553d4381d842dc88f70f3c0724b4da2c1e77a6ed7fbbdca56b29af19a
  Stored in directory: /tmp/pip-ephem-wheel-cache-009lt7i0/wheels/fd/89/93/70c1e5f378ee5043de89387ee3ef6852ff39e3b9eb44ecc1a3
Successfully built lightfm


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')


In [3]:
import os
import numpy as np
import pandas as pd
import random
from tqdm import tqdm

from scipy import sparse
from scipy.sparse import csr_matrix

from lightfm import LightFM
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MultiLabelBinarizer

import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
random.seed(42)

In [4]:
header = "drive/MyDrive"

STEAM_META_PATH = os.path.join("steam.csv")
STEAM_DESC_PATH = os.path.join("steam_description_data.csv")
TRAIN_PATH = os.path.join("data", "split", "train_split.csv")
VAL_PATH = os.path.join("data", "split", "val_split.csv")
TEST_PATH = os.path.join("data", "split", "test_split.csv")
METADATA_JSON_PATH = os.path.join("games_metadata.json")

La idea ahora es cargar todos los csvs.

In [5]:

def load_data():
    steam = pd.read_csv(STEAM_META_PATH)
    desc  = pd.read_csv(STEAM_DESC_PATH)
    train = pd.read_csv(TRAIN_PATH)
    val   = pd.read_csv(VAL_PATH)
    test  = pd.read_csv(TEST_PATH)
    metadata_json = pd.read_json(METADATA_JSON_PATH, lines=True)

    # normalizar is_recommended
    def to_bool(x):
        if isinstance(x, str):
            return x.lower() == "true"
        return bool(x)

    for df in (train, val, test):
        if "is_recommended" not in df.columns:
            df["is_recommended"] = False
        df["is_recommended"] = df["is_recommended"].apply(to_bool)
        df["hours"] = df["hours"].clip(lower=0.0)

    return steam, desc, train, val, test, metadata_json


steam, desc, train, val, test, metadata_json = load_data()

Ahora mapeamos correctamente los usuarios e items para que Light pueda manejarlo sin problemas

In [6]:

def prepare_items(steam, desc, app_ids):
    steam["genres"] = steam["genres"].fillna("").apply(lambda x: x.split(";"))
    desc = desc.rename(columns={"steam_appid": "appid"})
    items = steam[steam.appid.isin(app_ids)].merge(desc, on="appid", how="left")
    items = items.drop_duplicates("appid").reset_index(drop=True)
    return items

all_app_ids = pd.concat([train.app_id, val.app_id, test.app_id]).unique()
items = prepare_items(steam, desc, all_app_ids)

def build_id_maps(train, val, test, items):
    users = np.sort(pd.concat([train.user_id, val.user_id, test.user_id]).unique())
    apps  = np.sort(items.appid.unique())
    user_id_map = {u: i for i, u in enumerate(users)}
    item_id_map = {a: i for i, a in enumerate(apps)}
    return user_id_map, item_id_map, users, apps

user_id_map, item_id_map, visible_users, visible_items = build_id_maps(train, val, test, items)

items_ordered = (
    items.set_index("appid")
         .loc[np.sort(list(item_id_map.keys()))]
         .reset_index()
)


Se usó sbert, basado en el práctico del curso y en la documentación
encontrada en https://www.sbert.net/

In [7]:
def build_sbert_embeddings(items_ordered, batch_size=64, device="cuda"):
    model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

    text = (
        items_ordered["short_description"].fillna("") + " " +
        items_ordered["about_the_game"].fillna("") + " " +
        items_ordered["detailed_description"].fillna("")
    ).tolist()

    emb = model.encode(
        text,
        batch_size=batch_size,
        convert_to_numpy=True,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    emb = sparse.csr_matrix(emb.astype(np.float32))
    print("SBERT embeddings:", emb.shape)
    return emb

item_text_emb = build_sbert_embeddings(items_ordered, device="cuda")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/30 [00:00<?, ?it/s]

SBERT embeddings: (1872, 384)


Aquí usamos MultiLabelBinarizer, es algo que se encuentra en Scikit y nos basamos en https://www.kdnuggets.com/2023/01/encoding-categorical-features-multilabelbinarizer.html y en https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MultiLabelBinarizer.html

In [8]:
def split_tokens(series):
    return series.fillna("").apply(lambda x: [t.strip() for t in str(x).split(";") if t.strip()])

def build_categorical_features(items, top_cat=50):
    genres_mlb = MultiLabelBinarizer()
    genres = genres_mlb.fit_transform(split_tokens(items.genres))
    genres = sparse.csr_matrix(genres.astype(np.float32))

    cat_tokens = split_tokens(items.categories)
    all_cats = [c for lst in cat_tokens for c in lst]
    top = set(pd.Series(all_cats).value_counts().head(top_cat).index)

    cat_tokens_f = cat_tokens.apply(lambda lst: [c for c in lst if c in top])
    cats_mlb = MultiLabelBinarizer()
    categories = cats_mlb.fit_transform(cat_tokens_f)
    categories = sparse.csr_matrix(categories.astype(np.float32))

    return genres, categories

genres_mat, categories_mat = build_categorical_features(items_ordered)
item_features = sparse.hstack(
    [genres_mat, categories_mat, item_text_emb],
    format="csr",
    dtype=np.float32
)



Ya podemos generar las interacciones. Al igual que en el resto de modelos, consideramos que lo relevante es is_recommended = True

In [9]:
def build_interactions(df, user_id_map, item_id_map, mode="train"):

    df = df[df["app_id"].isin(item_id_map.keys())]

    if mode == "train":
        user_idx = df["user_id"].map(user_id_map)
        item_idx = df["app_id"].map(item_id_map)
        valid = user_idx.notnull() & item_idx.notnull()

        data = np.log1p(df.loc[valid, "hours"].values).astype(np.float32)
        return sparse.csr_matrix(
            (data, (user_idx[valid], item_idx[valid])),
            shape=(len(user_id_map), len(item_id_map)),
        )

    if mode == "eval":
        df2 = df.copy()
        mh = df2.groupby("user_id")["hours"].max()
        df2 = df2.merge(mh.rename("max_h"), left_on="user_id", right_index=True)

        user_idx = df2["user_id"].map(user_id_map)
        item_idx = df2["app_id"].map(item_id_map)
        valid = user_idx.notnull() & item_idx.notnull()

        data = np.ones(valid.sum(), dtype=np.float32)
        return sparse.csr_matrix(
            (data, (user_idx[valid], item_idx[valid])),
            shape=(len(user_id_map), len(item_id_map)),
        )

interactions_train = build_interactions(train, user_id_map, item_id_map, mode="train")
interactions_val   = build_interactions(val,   user_id_map, item_id_map, mode="eval")
interactions_test  = build_interactions(test,  user_id_map, item_id_map, mode="eval")

Generamos los features de usuario, para poder continuar.

In [10]:
def build_user_features(interactions, genres, categories):
    genres_bin = genres.astype(bool).astype(np.float32)
    cats_bin = categories.astype(bool).astype(np.float32)

    user_gen = interactions.dot(genres_bin)
    user_cat = interactions.dot(cats_bin)

    user_gen.data[:] = 1.0
    user_cat.data[:] = 1.0

    return sparse.hstack([user_gen, user_cat], format="csr", dtype=np.float32)

user_features = build_user_features(interactions_train, genres_mat, categories_mat)

Al igual que en los otros modelos, is_recommended True es lo relevante

In [11]:
def build_rel_idx_from_isrec(df, user_id_map, item_id_map):
    df2 = df[df["app_id"].isin(item_id_map.keys())].copy()
    df2["user_idx"] = df2["user_id"].map(user_id_map)
    df2["item_idx"] = df2["app_id"].map(item_id_map)

    all_users = df2["user_idx"].unique()

    df_rel = df2[df2["is_recommended"] == True]
    rel_true = df_rel.groupby("user_idx")["item_idx"].apply(set).to_dict()

    return {u: rel_true.get(u, set()) for u in all_users}

val_rel_idx  = build_rel_idx_from_isrec(val,  user_id_map, item_id_map)
test_rel_idx = build_rel_idx_from_isrec(test, user_id_map, item_id_map)

Ahora, definimos las métricas del mismo modo que en todo el proyecto.

In [12]:
def precision_at_k(rec_k, rel):
  return sum(i in rel for i in rec_k)/len(rec_k) if rec_k else 0

def recall_at_k(rec_k, rel):
  return sum(i in rel for i in rec_k)/len(rel) if rel else 0

def ndcg_at_k(rec_k, rel):
    if not rel: return 0.0
    dcg = sum(1/np.log2(r+2) for r,i in enumerate(rec_k) if i in rel)
    ideal = min(len(rel), len(rec_k))
    idcg = sum(1/np.log2(r+2) for r in range(ideal))
    return dcg/idcg if idcg > 0 else 0

def hit_at_k(rec_k, rel):
  return 1.0 if any(i in rel for i in rec_k) else 0

def map_at_k(rec_k, rel):
    if not rel: return 0
    hits, ap = 0, 0
    for r,it in enumerate(rec_k, start=1):
        if it in rel:
            hits += 1
            ap += hits/r
    return ap/len(rel)

def diversity_at_k(rec_dict, info_videojuegos):
    vals=[]
    for uid, recs in rec_dict.items():
        gs=set()
        for it in recs:
            if it in info_videojuegos:
                gs.update(info_videojuegos[it][1])
        vals.append(len(gs))
    return np.mean(vals)

También rescatamos info de los juegos, para calcular la diversidad

In [14]:
# info_videojuegos = {
#     item_id_map[row.appid]: (row.appid, row.genres)
#     for _, row in items.iterrows()
#     if isinstance(row.genres, list)
# }

#sólo para el cambio en diversity
info_videojuegos = {
    item_id_map[row.app_id]: (row.app_id, row.tags)
    for _, row in metadata_json.iterrows()
    if isinstance(row.tags, list) and row.app_id in item_id_map
}

Además, filtramos items vistos, lo que es clave para lo visto.

In [15]:
train_seen = {
    u: set(interactions_train[u].indices)
    for u in range(interactions_train.shape[0])
}


Ahora definimos una función para evaluar el modelo. Opcionalmente, para hacer pruebas rápidas, usamos samples de usuarios, pero luego la idea es evaluar con todos.

In [16]:
def evaluate_lightfm(model, rel_idx, sample=None):

    n_users, n_items = interactions_train.shape

    users = list(rel_idx.keys())
    if sample:
        users = random.sample(users, min(sample, len(users)))

    rec_dict = {}
    P, R, N, H, M = [], [], [], [], []

    for u in users:

        item_ids = np.arange(n_items, dtype=np.int32)
        user_ids = np.repeat(u, n_items).astype(np.int32)

        scores = model.predict(
            user_ids,
            item_ids,
            item_features=item_features,
            user_features=user_features,
            num_threads=4
        )

        # filtrar vistos
        seen = train_seen.get(u, set())
        if seen:
            scores[list(seen)] = -1e9

        # top-10
        topk = np.argpartition(-scores, 10)[:10]
        topk = topk[np.argsort(-scores[topk])]
        topk = topk.tolist()

        rec_dict[u] = topk
        rel = rel_idx[u]

        P.append(precision_at_k(topk, rel))
        R.append(recall_at_k(topk, rel))
        N.append(ndcg_at_k(topk, rel))
        H.append(hit_at_k(topk, rel))
        M.append(map_at_k(topk, rel))

    diversity = diversity_at_k(rec_dict, info_videojuegos)

    out = {
        "P@10": np.mean(P),
        "R@10": np.mean(R),
        "F1@10": 2 * (np.mean(P) * np.mean(R)) / (np.mean(P) + np.mean(R)),
        "NDCG@10": np.mean(N),
        "HIT@10": np.mean(H),
        "MAP@10": np.mean(M),
        "Diversity@10": diversity,
    }

    return out

Finalmente, la idea es hacer un random search con 30 iteraciones sobre un conjunto amplio de hiperparámetros para buscar una combinación óptima.

Se usó warp y warp-koss como losses potenciales. Esta última optimiza directamente el ranking en el top-K, enfocando las actualizaciones en los errores que afectan los primeros puestos de la lista. Según Kula (2015), esta variante mejora recall y ndcg cuando hay señales implícitas y item features ricos (como SBERT), superando a BPR en escenarios con contenido semántico. Por lo tanto, usamos esas dos para probar.

In [17]:
param_space = {
    "loss": ["warp", "warp-kos"],
    "no_components": [32, 48, 64, 96],
    "learning_rate": [0.005, 0.01, 0.02],
    "epochs": [5, 8, 12],
    "item_alpha": [0.0, 1e-6, 1e-5],
    "user_alpha": [0.0, 1e-6, 1e-5],
}

random.seed(3533)

def sample_params():
    return {
        "loss": random.choice(param_space["loss"]),
        "no_components": random.choice(param_space["no_components"]),
        "learning_rate": random.choice(param_space["learning_rate"]),
        "epochs": random.choice(param_space["epochs"]),
    }

best = None
results = []
print("empezamos")
for i in range(30):
    print(f"\n iter {i+1}/30")
    params = sample_params()
    print(params)

    model = LightFM(
        loss=params["loss"],
        no_components=params["no_components"],
        learning_rate=params["learning_rate"],
        random_state=42,
    )

    model.fit(
        interactions_train,
        item_features=item_features,
        user_features=user_features,
        epochs=params["epochs"],
        num_threads=4,
        verbose=False,
    )

    report = evaluate_lightfm(model, val_rel_idx, sample=800)
    score = report["R@10"]

    results.append({**params, **report})

    if best is None or score > best["R@10"]:
        best = {**params, **report, "model": model}



empezamos

 iter 1/30
{'loss': 'warp', 'no_components': 48, 'learning_rate': 0.02, 'epochs': 8}

 iter 2/30
{'loss': 'warp', 'no_components': 48, 'learning_rate': 0.01, 'epochs': 12}

 iter 3/30
{'loss': 'warp-kos', 'no_components': 48, 'learning_rate': 0.02, 'epochs': 8}

 iter 4/30
{'loss': 'warp-kos', 'no_components': 48, 'learning_rate': 0.005, 'epochs': 5}

 iter 5/30
{'loss': 'warp-kos', 'no_components': 64, 'learning_rate': 0.02, 'epochs': 8}

 iter 6/30
{'loss': 'warp-kos', 'no_components': 32, 'learning_rate': 0.005, 'epochs': 8}

 iter 7/30
{'loss': 'warp', 'no_components': 96, 'learning_rate': 0.01, 'epochs': 8}

 iter 8/30
{'loss': 'warp-kos', 'no_components': 48, 'learning_rate': 0.01, 'epochs': 12}

 iter 9/30
{'loss': 'warp', 'no_components': 64, 'learning_rate': 0.02, 'epochs': 8}

 iter 10/30
{'loss': 'warp', 'no_components': 64, 'learning_rate': 0.01, 'epochs': 12}

 iter 11/30
{'loss': 'warp-kos', 'no_components': 48, 'learning_rate': 0.02, 'epochs': 5}

 iter 12/30


Con la mejor combinación, ya evaluamos en test

In [18]:
print("\nmejor config")
print(best)

print("\nev final en test")
test_report = evaluate_lightfm(best["model"], test_rel_idx, sample=None)
print(test_report)


mejor config
{'loss': 'warp', 'no_components': 48, 'learning_rate': 0.005, 'epochs': 5, 'P@10': np.float64(0.007749999999999999), 'R@10': np.float64(0.0721577380952381), 'F1@10': np.float64(0.01399670378785057), 'NDCG@10': np.float64(0.038455310912876614), 'HIT@10': np.float64(0.0775), 'MAP@10': np.float64(0.027583333333333328), 'Diversity@10': np.float64(32.925), 'model': <lightfm.lightfm.LightFM object at 0x7c5a4c4607a0>}

ev final en test
{'P@10': np.float64(0.005540435458786936), 'R@10': np.float64(0.050042120269569716), 'F1@10': np.float64(0.009976336422146806), 'NDCG@10': np.float64(0.025144093062544975), 'HIT@10': np.float64(0.05501555209953344), 'MAP@10': np.float64(0.01700675312893431), 'Diversity@10': np.float64(34.7348367029549)}
